Target transactions: We reuse Lab Five’s prepared inputs and labels. “For each target” means “for each transaction represented by a row in Lab Five’s X_train or X_validation.” These are the transactions whose laundering risk we want to predict.

Historical transactions: We use a broader table covering September 1–8. For each target, we select up to eight transactions sent by the same account in the preceding 24 hours, strictly before the target timestamp.

We prepare those earlier events as a separate sequence input. The GRU turns that sequence into 16 summary values, which we combine with the target transaction’s existing 44 features.

A transaction can play b

In [5]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"
cache_dir = project_root / "data/processed/lab09"
cache_dir.mkdir(parents=True, exist_ok=True)

In [6]:
sample = pd.read_csv(csv_path, nrows=100_000)

total_rows = 5_078_345
sample_mib = sample.memory_usage(deep=True).sum() / 1024**2
estimated_mib = sample_mib * total_rows / len(sample)

print(f"Sample: {sample_mib:.1f} MiB")
print(f"Estimated full table: {estimated_mib:.1f} MiB")

Sample: 14.4 MiB
Estimated full table: 729.5 MiB


In [7]:
def load_sequence_events(path: Path) -> pd.DataFrame:
    """Load transaction history without laundering labels."""
    columns = [
        "timestamp",
        "sender_bank",
        "sender_account",
        "amount_paid",
        "payment_currency",
        "payment_format",
    ]
    string_columns = [
        "sender_bank",
        "sender_account",
        "payment_currency",
        "payment_format",
    ]

    parts = []
    row_offset = 0

    for chunk in iter_transaction_chunks(path, usecols=columns):
        # Preserve the original CSV row identity before filtering.
        chunk["source_row"] = range(
            row_offset, row_offset + len(chunk)
        )
        row_offset += len(chunk)

        chunk["timestamp"] = pd.to_datetime(
            chunk["timestamp"],
            format="%Y/%m/%d %H:%M",
            errors="raise",
        )

        in_period = (
            chunk["timestamp"].ge("2022-09-01")
            & chunk["timestamp"].lt("2022-09-09")
        )
        selected = chunk.loc[in_period].copy()

        if selected.empty:
            continue

        for column in string_columns:
            selected[column] = selected[column].astype(
                "string[pyarrow]"
            )

        parts.append(selected)

    events = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["timestamp", "source_row"])
        .reset_index(drop=True)
    )
    parts.clear()

    for column in ["payment_currency", "payment_format"]:
        events[column] = events[column].astype("category")

    return events

In [8]:
events = load_sequence_events(csv_path)

assert events["source_row"].is_unique
assert events["timestamp"].is_monotonic_increasing

events.to_parquet(
    cache_dir / "sequence_events.parquet",
    index=False,
)

memory_mib = events.memory_usage(deep=True).sum() / 1024**2

print(f"History transactions: {len(events):,}")
print(f"DataFrame memory: {memory_mib:.1f} MiB")
display(events.head())

History transactions: 4,214,445
DataFrame memory: 227.7 MiB


,timestamp,sender_bank,sender_account,amount_paid,payment_currency,payment_format,source_row
0,2022-09-01,03209,8000F4670,14675.57,US Dollar,Reinvestment,2
1,2022-09-01,01420,8005DFEB0,897.37,US Dollar,Reinvestment,17
2,2022-09-01,010,8000F6850,99986.94,US Dollar,Reinvestment,158
3,2022-09-01,012,8001167D0,16.08,US Dollar,Reinvestment,218
4,2022-09-01,001,8001364A0,10.30,US Dollar,Reinvestment,281


In [10]:
in_training = (
    events["timestamp"].ge("2022-09-03")
    & events["timestamp"].lt("2022-09-07")
)

account_counts = (
    events.loc[in_training]
    .groupby(["sender_bank", "sender_account"])
    .size()
)

example_bank, example_account = account_counts.idxmax()

same_account = (
    events["sender_bank"].eq(example_bank)
    & events["sender_account"].eq(example_account)
)
account_events = events.loc[same_account].copy()

account_training = account_events.loc[
    account_events["timestamp"].ge("2022-09-03")
    & account_events["timestamp"].lt("2022-09-07")
]

# The transaction whose available history we want to inspect.
target = account_training.iloc[-1]
target_time = target["timestamp"]

available_history = account_events.loc[
    account_events["timestamp"].ge(
        target_time - pd.Timedelta(hours=24)
    )
    & account_events["timestamp"].lt(target_time)
]

prior_events = available_history.tail(8).copy()
prior_events["minutes_before_target"] = (
    target_time - prior_events["timestamp"]
).dt.total_seconds() / 60

display_columns = [
    "source_row",
    "timestamp",
    "amount_paid",
    "payment_currency",
    "payment_format",
]

print("Transaction to score:")
display(target[display_columns].to_frame().T)

print("Earlier transactions available to the model:")
display(
    prior_events[
        display_columns + ["minutes_before_target"]
    ]
)

Transaction to score:


,source_row,timestamp,amount_paid,payment_currency,payment_format
3248708,3242895,2022-09-06 23:59:00,73267.16,US Dollar,Cash


Earlier transactions available to the model:


,source_row,timestamp,amount_paid,payment_currency,payment_format,minutes_before_target
3248290,3240822,2022-09-06 23:58:00,897705.29,US Dollar,Cash,1.0
3248304,3241293,2022-09-06 23:58:00,58.74,US Dollar,Cash,1.0
3248305,3241317,2022-09-06 23:58:00,3.10,US Dollar,Cheque,1.0
3248308,3241404,2022-09-06 23:58:00,309.02,US Dollar,Cheque,1.0
3248320,3241708,2022-09-06 23:58:00,180.84,US Dollar,Cheque,1.0
3248351,3242530,2022-09-06 23:58:00,197.02,US Dollar,Cheque,1.0
3248354,3242611,2022-09-06 23:58:00,579.33,US Dollar,Cheque,1.0
3248387,3243756,2022-09-06 23:58:00,619.69,US Dollar,Cheque,1.0


In [11]:
# test = account_training.iloc[-3:,:]
# testAllow notices from a Cinder account.

In [12]:
assert 0 < len(prior_events) <= 8

assert prior_events["timestamp"].lt(target_time).all()
assert prior_events["timestamp"].ge(
    target_time - pd.Timedelta(hours=24)
).all()

assert prior_events["sender_bank"].eq(example_bank).all()
assert prior_events["sender_account"].eq(example_account).all()

assert target["source_row"] not in prior_events["source_row"].values
assert prior_events["timestamp"].is_monotonic_increasing

print("History checks passed.")

History checks passed.


In [18]:
import numpy as np


def build_history_index(
    events: pd.DataFrame,
    target_source_rows: np.ndarray,
    *,
    max_events: int = 8,
    hours: int = 24,
) -> tuple[np.ndarray, np.ndarray]:
    """Return earlier-event row positions and history lengths."""
    if max_events < 1 or hours <= 0:
        raise ValueError("History size and window must be positive.")

    if not events["source_row"].is_unique:
        raise ValueError("Event source rows must be unique.")

    required = ["sender_bank", "sender_account", "timestamp"]
    if events[required].isna().any().any():
        raise ValueError("History needs complete account IDs and timestamps.")

    # Locate each target in the existing event table.
    target_positions = pd.Index(
        events["source_row"]
    ).get_indexer(target_source_rows)

    if (target_positions < 0).any():
        raise ValueError("Some target transactions are missing from events.")

    # Temporary grouping IDs, not learned embeddings or model inputs.
    account_ids, _ = pd.factorize(
        pd.MultiIndex.from_frame(
            events[["sender_bank", "sender_account"]]
        )
    )
    times = events["timestamp"].to_numpy(dtype="datetime64[ns]")

    # lexsort uses its last key as the primary sorting key.
    order = np.lexsort((
        events["source_row"].to_numpy(),
        times,
        account_ids,
    ))

    # Search using account + time together, so accounts stay separate.
    keys = np.rec.fromarrays(
        [account_ids[order], times[order]],
        names=["account", "time"],
    )
    target_keys = np.rec.fromarrays(
        [account_ids[target_positions], times[target_positions]],
        names=["account", "time"],
    )

    window_start = target_keys.copy()
    window_start["time"] -= np.timedelta64(hours, "h")

    starts = np.searchsorted(keys, window_start, side="left")
    ends = np.searchsorted(keys, target_keys, side="left")

    lengths = np.minimum(ends - starts, max_events).astype(np.int64)

    # Real history occupies the first slots; padding follows it.
    slots = np.arange(max_events)
    valid = slots[None, :] < lengths[:, None]

    positions = ends[:, None] - lengths[:, None] + slots[None, :]
    positions[~valid] = 0  # Safe temporary position for the lookup.

    history_index = order[positions].astype(np.int32)
    history_index[~valid] = -1

    return history_index, lengths

In [14]:
previous_cache = project_root / "data/processed/lab05"

train_source_rows = np.load(
    previous_cache / "train_source_rows.npy"
)
validation_source_rows = np.load(
    previous_cache / "validation_source_rows.npy"
)

target_source_rows = np.concatenate([
    train_source_rows,
    validation_source_rows,
])

history_index, history_lengths = build_history_index(
    events,
    target_source_rows,
)

n_train = len(train_source_rows)

train_history_index = history_index[:n_train]
validation_history_index = history_index[n_train:]

train_history_lengths = history_lengths[:n_train]
validation_history_lengths = history_lengths[n_train:]

print("Training shape:", train_history_index.shape)
print("Validation shape:", validation_history_index.shape)
print(f"Index memory: {history_index.nbytes / 1024**2:.1f} MiB")

Training shape: (1379551, 8)
Validation shape: (965524, 8)
Index memory: 71.6 MiB


In [15]:
coverage = pd.DataFrame([
    {
        "split": name,
        "no_history": int((lengths == 0).sum()),
        "one_to_seven": int(
            ((lengths > 0) & (lengths < 8)).sum()
        ),
        "eight": int((lengths == 8).sum()),
    }
    for name, lengths in [
        ("train", train_history_lengths),
        ("validation", validation_history_lengths),
    ]
])

display(coverage)

,split,no_history,one_to_seven,eight
0,train,190949,809183,379419
1,validation,93947,537714,333863


In [ ]:
example_position = pd.Index(
    target_source_rows
).get_loc(target["source_row"])

example_length = int(history_lengths[example_position])
example_rows = history_index[
    example_position, :example_length
]

indexed_history = events.iloc[example_rows]

np.testing.assert_array_equal(
    indexed_history["source_row"].to_numpy(),
    prior_events["source_row"].to_numpy(),
)

assert history_index.shape == (len(target_source_rows), 8)
assert ((history_lengths >= 0) & (history_lengths <= 8)).all()

valid_slots = (
    np.arange(8)[None, :] < history_lengths[:, None]
)
assert (history_index[~valid_slots] == -1).all()
assert (history_index[valid_slots] >= 0).all()

print("Indexed history matches the Step 1 example.")
display(indexed_history[display_columns])

Indexed history matches the Step 1 example.


,source_row,timestamp,amount_paid,payment_currency,payment_format
3248290,3240822,2022-09-06 23:58:00,897705.29,US Dollar,Cash
3248304,3241293,2022-09-06 23:58:00,58.74,US Dollar,Cash
3248305,3241317,2022-09-06 23:58:00,3.10,US Dollar,Cheque
3248308,3241404,2022-09-06 23:58:00,309.02,US Dollar,Cheque
3248320,3241708,2022-09-06 23:58:00,180.84,US Dollar,Cheque
3248351,3242530,2022-09-06 23:58:00,197.02,US Dollar,Cheque
3248354,3242611,2022-09-06 23:58:00,579.33,US Dollar,Cheque
3248387,3243756,2022-09-06 23:58:00,619.69,US Dollar,Cheque


In [24]:
arrays_to_save = {
    "train_history_index": train_history_index,
    "validation_history_index": validation_history_index,
    "train_history_lengths": train_history_lengths,
    "validation_history_lengths": validation_history_lengths,
    "train_source_rows": train_source_rows,
    "validation_source_rows": validation_source_rows,
    "event_source_rows": events["source_row"].to_numpy(),
}

for name, values in arrays_to_save.items():
    np.save(cache_dir / f"{name}.npy", values)

print("History indices and row identities saved.")

History indices and row identities saved.


In [25]:
import joblib
import torch
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn

from src.neural import NeuralBaseline

torch.set_num_threads(2)

# Row positions must still refer to the same event-table order.
np.testing.assert_array_equal(
    events["source_row"].to_numpy(),
    np.load(cache_dir / "event_source_rows.npy"),
)

row_lookup = pd.Index(events["source_row"])
train_positions = row_lookup.get_indexer(train_source_rows)
assert (train_positions >= 0).all()

event_columns = ["payment_currency", "payment_format"]

event_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32,
)
event_encoder.fit(
    events.iloc[train_positions][event_columns]
)

amount_scaler = StandardScaler()
amount_scaler.fit(
    np.log1p(
        events.iloc[train_positions][["amount_paid"]].to_numpy()
    )
)

n_event_features = 1 + sum(
    len(categories)
    for categories in event_encoder.categories_
)

# Store each event's features once.
event_features = np.empty(
    (len(events), n_event_features),
    dtype=np.float32,
)

# Transform in chunks to limit temporary allocations.
for start in range(0, len(events), 100_000):
    stop = min(start + 100_000, len(events))
    part = events.iloc[start:stop]

    event_features[start:stop, :1] = amount_scaler.transform(
        np.log1p(part[["amount_paid"]].to_numpy())
    )
    event_features[start:stop, 1:] = event_encoder.transform(
        part[event_columns]
    )

event_times = events["timestamp"].to_numpy(
    dtype="datetime64[ns]"
)

print("Event features:", event_features.shape)
print(f"Memory: {event_features.nbytes / 1024**2:.1f} MiB")

Event features: (4214445, 22)
Memory: 353.7 MiB


In [26]:
split_data = {}

for name in ["train", "validation"]:
    source_rows = np.load(
        cache_dir / f"{name}_source_rows.npy"
    )

    np.testing.assert_array_equal(
        source_rows,
        np.load(previous_cache / f"{name}_source_rows.npy"),
    )

    positions = row_lookup.get_indexer(source_rows)
    assert (positions >= 0).all()

    split_data[name] = {
        "current": np.load(
            previous_cache / f"X_{name}.npy",
            mmap_mode="r",
        ),
        "labels": np.load(
            previous_cache / f"y_{name}.npy"
        ),
        "history_index": np.load(
            cache_dir / f"{name}_history_index.npy",
            mmap_mode="r",
        ),
        "lengths": np.load(
            cache_dir / f"{name}_history_lengths.npy",
            mmap_mode="r",
        ),
        "times": event_times[positions],
    }

    assert all(
        len(values) == len(source_rows)
        for values in split_data[name].values()
    ) 

In [ ]:
def make_sequence_batch(
    rows,
    *,
    split,
    event_features,
    event_times,
):
    """Gather current inputs, historical events, lengths, and labels."""
    indices = split["history_index"][rows]
    valid = indices >= 0

    # Never use padding ID -1 directly as an array index.
    safe_indices = np.where(valid, indices, 0)
    history = event_features[safe_indices]

    age_minutes = (
        split["times"][rows, None] - event_times[safe_indices]
    ) / np.timedelta64(1, "m")

    age_minutes = np.where(valid, age_minutes, 0.0)

    if (
        (age_minutes[valid] <= 0).any()
        or (age_minutes[valid] > 1440).any()
    ):
        raise ValueError(
            "History must be earlier and within 24 hours."
        )

    # Map ages in the 24-hour window to a compact numerical scale.
    age = np.log1p(age_minutes) / np.log1p(1440.0)

    history = np.concatenate(
        [history, age.astype(np.float32)[..., None]],
        axis=-1,
    )
    history[~valid] = 0.0

    return (
        torch.from_numpy(
            np.array(
                split["current"][rows],
                dtype=np.float32,
                copy=True,
            )
        ),
        torch.from_numpy(history),
        torch.from_numpy(split["lengths"][rows].copy()),
        torch.from_numpy(split["labels"][rows].copy()),
    )

In [28]:
class SequenceClassifier(nn.Module):
    def __init__(
        self,
        n_current,
        n_event,
        *,
        positive_rate,
        hidden_size=16,
        use_history=True,
    ):
        super().__init__()

        self.use_history = use_history
        self.hidden_size = hidden_size

        self.classifier = NeuralBaseline(
            n_current + hidden_size,
            positive_rate=positive_rate,
        )
        self.gru = nn.GRU(
            input_size=n_event,
            hidden_size=hidden_size,
            batch_first=True,
        )

    def forward(self, current, history, lengths):
        context = current.new_zeros(
            (len(current), self.hidden_size)
        )

        if self.use_history:
            outputs, _ = self.gru(history)

            # Select the output after the last real event.
            last = (lengths - 1).clamp_min(0)
            context = outputs[
                torch.arange(len(current)), last
            ]

            # Empty histories receive an explicit zero vector.
            context = context * (lengths > 0).unsqueeze(1)

        combined = torch.cat([current, context], dim=1)
        return self.classifier(combined)

In [29]:
train_data = split_data["train"]

example_rows = np.array([
    np.flatnonzero(train_data["lengths"] == length)[0]
    for length in [0, 3, 8]
])

current, history, lengths, labels = make_sequence_batch(
    example_rows,
    split=train_data,
    event_features=event_features,
    event_times=event_times,
)

torch.manual_seed(42)

model = SequenceClassifier(
    n_current=current.shape[1],
    n_event=history.shape[2],
    positive_rate=float(train_data["labels"].mean()),
)

model.eval()

with torch.inference_mode():
    logits = model(current, history, lengths)

print("Current:", tuple(current.shape))
print("History:", tuple(history.shape))
print("Lengths:", lengths.tolist())
print("Logits:", tuple(logits.shape))

assert logits.shape == (3,)
assert torch.isfinite(logits).all()

Current: (3, 44)
History: (3, 8, 23)
Lengths: [0, 3, 8]
Logits: (3,)


In [ ]:
# Now deliberately corrupt the padding.
# Predictions should remain unchanged.
padding = (
    torch.arange(history.shape[1])[None, :]
    >= lengths[:, None]
)

changed_history = history.clone()
changed_history[padding] = 123.0

with torch.inference_mode():
    changed_logits = model(
        current, changed_history, lengths
    )

torch.testing.assert_close(changed_logits, logits)

print("Padding does not change predictions.")

Padding does not change predictions.


In [31]:
np.save(cache_dir / "event_features.npy", event_features)

joblib.dump(
    {
        "encoder": event_encoder,
        "amount_scaler": amount_scaler,
        "categorical_columns": event_columns,
    },
    cache_dir / "event_preprocessing.joblib",
)

['/Users/sara/Projects/data-science-lab/data/processed/lab09/event_preprocessing.joblib']